# Mashi TTS — Fine-tune XTTS v2

Fine-tunes Coqui XTTS v2 on the Mashi 3-speaker dataset (723 clips, 67 min).

**Before running:** upload `data/mashi_dataset/` to Google Drive under
`My Drive/mashi-tts-bootstrap/data/mashi_dataset/`. It must contain
`audio/`, `metadata_train.csv`, `metadata_eval.csv`, `speaker_references/`
(produced by scripts 05 and 10).

**Language proxy:** XTTS v2 has no Bantu language, so we train through the
Spanish (`es`) frontend — 5 pure vowels and open CV syllables make it the
closest phonetic proxy for Mashi. The `es` text cleaner also expands the
clock-time digits ("9:05") the same way at training and inference, keeping
them consistent.

Runtime: GPU (T4 is enough). ~1-2 h for 10 epochs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/mashi-tts-bootstrap'
SEG_DIR    = f'{DRIVE_ROOT}/data/mashi_dataset'
MODEL_DIR  = f'{DRIVE_ROOT}/models/checkpoints'
LANGUAGE   = 'es'   # phonetic proxy — XTTS v2 has no Bantu language

import os
os.makedirs(MODEL_DIR, exist_ok=True)
assert os.path.exists(f'{SEG_DIR}/metadata_train.csv'), 'upload data/mashi_dataset to Drive first'

In [ ]:
# coqui-tts is the maintained fork of Coqui TTS — works on Colab's Python 3.12
# (the original TTS==0.22.0 package only supports Python <3.12)
!pip install -q coqui-tts

import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'  # TTS checkpoints predate torch 2.6 safe-loading

In [ ]:
# Fine-tune the GPT part of XTTS v2 on the Mashi clips
from TTS.demos.xtts_ft_demo.utils.gpt_train import train_gpt

config_path, original_ckpt, vocab_file, exp_path, speaker_ref = train_gpt(
    language=LANGUAGE,
    num_epochs=10,
    batch_size=4,
    grad_acumm=4,                                # effective batch 16
    train_csv=f'{SEG_DIR}/metadata_train.csv',
    eval_csv=f'{SEG_DIR}/metadata_eval.csv',
    output_path=MODEL_DIR,
    max_audio_length=255995,                     # 11.6 s at 22050 Hz — longest clip is 11.3 s
)

ft_checkpoint = os.path.join(exp_path, 'best_model.pth')
print('config :', config_path)
print('vocab  :', vocab_file)
print('ckpt   :', ft_checkpoint)

In [ ]:
# Load the fine-tuned model
import torch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

config = XttsConfig()
config.load_json(config_path)
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_path=ft_checkpoint, vocab_path=vocab_file, use_deepspeed=False)
model.cuda()
print('model loaded')

In [ ]:
# Generate test samples — one block per speaker, conditioned on that speaker's reference wav
import torchaudio, os

tests = {
    'bib': ["Nnâmahanga alema amalunga n'igulu.",
            "Obulangashane bwanaciba."],
    'dav': ["Mbwira ngahi abantu barhengaga?",
            "Muguma, babirhi, basharhu, bani, barhanu."],
    'mrl': ["9:05",
            "16:35"],
}

out_dir = f'{DRIVE_ROOT}/results/samples'
os.makedirs(out_dir, exist_ok=True)

for spk, sentences in tests.items():
    ref = f'{SEG_DIR}/speaker_references/ref_{spk}.wav'
    gpt_cond, speaker_emb = model.get_conditioning_latents(audio_path=[ref])
    for i, text in enumerate(sentences):
        out = model.inference(text, LANGUAGE, gpt_cond, speaker_emb, temperature=0.7)
        path = f'{out_dir}/{spk}_sample_{i+1:02d}.wav'
        torchaudio.save(path, torch.tensor(out['wav']).unsqueeze(0), 24000)
        print('saved', path)

## Notes

- **Listen in `results/samples/`** on Drive. Expect *intelligible* Mashi with the
  speaker's timbre, not production quality — 67 min is a bootstrap dataset.
- To try other voices, point `ref` at any training clip of that speaker.
- Times must be typed exactly like the training transcripts (`9:05`, `16:35`).
- If Colab runs out of memory, set `batch_size=2, grad_acumm=8` (same effective batch).